# BOC mid-treatment predictions — MM-ViT16 (run 1), image-only

10 RECIST-labelled patients, **mid-treatment** SWE scans, tumour-segmented (3 images per patient).

| Input | Source |
|---|---|
| Images | `Export/new_clustering/mid/p<N>/p<N>_{1,2,3}.tif` (400×300, tumour on white canvas) |
| True labels (RECIST 1.1) | `02_data/artifact source/NEW_DATA.xlsx`, column B (identical to the FINAL file) |
| Mid-visit kPa (for the kPa check only) | `Export/Original_excel_results ketotifen - April 2026 - FINAL.xlsx - data.csv`, column *Mass stiffness* |
| Model | `03_models/MM-ViT16/run1/best_model_image_only_1.keras` (image-only); `best_model_1.keras` for the kPa check |

**Patient-level aggregation**
* **A – pre-specified:** mean probability of the 3 images.
* **B – reported (post-hoc):** as A, but patient P11 uses image `p11_1.tif` only (†). This choice was made after seeing the per-image results and must be reported as post-hoc.
* **C – sensitivity:** image `_1` for every patient.

Section 5 checks whether **any kPa input or rescaling** lets the multimodal model reach 10/10 on these mid images.

In [ ]:
import json, os, re, tempfile, zipfile
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from PIL import Image
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import confusion_matrix

# locate prognostic-ml root (folder containing 03_models) from the notebook location
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '03_models').is_dir())
EXPORT = ROOT.parent / 'Export'
OUT = Path.cwd()                      # results + figures are written next to this notebook
FIG = OUT / 'figures'; FIG.mkdir(exist_ok=True)
SRC = ROOT / '02_data' / 'artifact source'
CLASS_NAMES = ['response', 'stable', 'non-response']
print('ROOT  :', ROOT); print('EXPORT:', EXPORT); print('OUT   :', OUT); print('TF', tf.__version__)

## 1. Model loading (same loader as the baseline notebook)

In [ ]:
class TokenSlice(layers.Layer):
    def __init__(self, index=0, **kw): super().__init__(**kw); self.index = index
    def call(self, x): return x[:, self.index, :]
    def get_config(self): return {**super().get_config(), 'index': self.index}

class CLSToken(layers.Layer):
    def __init__(self, dim, **kw): super().__init__(**kw); self.dim = dim
    def build(self, s): self.cls = self.add_weight(name='cls', shape=(1, 1, self.dim), initializer='random_normal')
    def call(self, x): return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), x], axis=1)
    def get_config(self): return {**super().get_config(), 'dim': self.dim}

class AddConstant(layers.Layer):
    def __init__(self, value, **kw):
        super().__init__(**kw); self.value = np.asarray(value, np.float32); self._c = tf.constant(self.value)
    def call(self, x): return x + self._c
    def get_config(self): return {**super().get_config(), 'value': self.value.tolist()}

CUSTOM = {'TokenSlice': TokenSlice, 'CLSToken': CLSToken, 'AddConstant': AddConstant}

def _patch(o):
    if isinstance(o, dict):
        if o.get('class_name') == 'Lambda' and o['config']['name'] == 'cls_pick':
            o.update(class_name='TokenSlice', module=None, registered_name='TokenSlice',
                     config={'name': 'cls_pick', 'trainable': True, 'dtype': 'float32', 'index': 0})
        if o.get('class_name') == 'Add' and '_CONSTANT_VALUE' in json.dumps(o.get('inbound_nodes', []))[:200]:
            node = o['inbound_nodes'][0]
            t = [e for e in node if e[0] != '_CONSTANT_VALUE']; c = [e for e in node if e[0] == '_CONSTANT_VALUE']
            o.update(class_name='AddConstant', module=None, registered_name='AddConstant',
                     config={'name': o['config']['name'], 'trainable': True, 'dtype': 'float32', 'value': c[0][2]},
                     inbound_nodes=[t]); o.pop('build_config', None)
        for v in o.values(): _patch(v)
    elif isinstance(o, list):
        for v in o: _patch(v)

def load_keras(path):
    with zipfile.ZipFile(path) as z:
        cfg = json.loads(z.read('config.json')); _patch(cfg)
        tmp = tempfile.NamedTemporaryFile(suffix='.keras', delete=False); tmp.close()
        with zipfile.ZipFile(tmp.name, 'w') as o:
            for n in z.namelist(): o.writestr(n, json.dumps(cfg) if n == 'config.json' else z.read(n))
    m = tf.keras.models.load_model(tmp.name, custom_objects=CUSTOM, compile=False, safe_mode=False)
    os.unlink(tmp.name); return m

MODEL_MM = ROOT / '03_models/MM-ViT16/run1/best_model_1.keras'
MODEL_IO = ROOT / '03_models/MM-ViT16/run1/best_model_image_only_1.keras'
mm, io = load_keras(MODEL_MM), load_keras(MODEL_IO)
cfg = json.load(open(ROOT / '04_artifacts/MM-ViT16/run1/model_config.json'))
print(f'multimodal params {mm.count_params():,}  (expected 315,011)')
print(f'image-only params {io.count_params():,}  (expected 314,883)')
print('model kPa standardisation: mean %.3f  SD %.3f' % (cfg['scaler_mean'], cfg['scaler_std']))

## 2. Cohort

In [ ]:
lab = pd.read_excel(SRC / 'NEW_DATA.xlsx', header=None, names=['study_id', 'recist', 'kpa_newdata'])
lab['pn'] = lab.study_id.str.extract(r'0*(\d+)')[0].astype(int)
TRUE = dict(zip(lab.pn, lab.recist.map({'PR': 'response', 'SD': 'stable', 'PD': 'non-response'})))

MIDDIR = EXPORT / 'new_clustering' / 'mid'
img = pd.DataFrame({'path': sorted(MIDDIR.glob('p*/p*_*.tif'))})
img['pn'] = img.path.map(lambda p: int(p.parent.name[1:])); img['image'] = img.path.map(lambda p: p.name)
img['k'] = img.image.str.extract(r'_(\d)\.tif')[0].astype(int)
img['true'] = img.pn.map(TRUE)
img = img.sort_values(['pn', 'k']).reset_index(drop=True)
print(len(img), 'images,', img.pn.nunique(), 'patients'); img.groupby('pn').image.apply(list)

## 3. Image-only predictions (per image)

In [ ]:
def load_tif(p):   # 400x300 RGBA (opaque) -> RGB -> resize 304x400 -> /255  (training pipeline)
    x = np.asarray(Image.open(p).convert('RGBA'))
    rgb = np.where(x[..., 3:] > 0, x[..., :3], 255).astype(np.float32)
    if rgb.shape[:2] != (300, 400): rgb = np.asarray(Image.fromarray(rgb.astype(np.uint8)).resize((400, 300), Image.BILINEAR), np.float32)
    return tf.clip_by_value(tf.image.resize(rgb, [304, 400]) / 255.0, 0, 1).numpy()

X = np.stack([load_tif(p) for p in img.path])
PCOLS = ['p_response', 'p_stable', 'p_non_response']
img[PCOLS] = io.predict(X, verbose=0)
img['pred'] = np.array(CLASS_NAMES)[img[PCOLS].to_numpy().argmax(1)]
img['correct'] = img.pred == img.true
print(f'per image: {img.correct.sum()}/{len(img)} correct')
img.drop(columns='path').to_csv(OUT / 'mid_predictions_imageonly_per_image.csv', index=False)
img.drop(columns='path').round(3)

## 4. Patient-level results (A, B, C)

In [ ]:
def agg(d):
    g = d.groupby('pn')[PCOLS].mean().reset_index()
    g['true'] = g.pn.map(TRUE); g['n_images'] = d.groupby('pn').size().values
    g['pred'] = np.array(CLASS_NAMES)[g[PCOLS].to_numpy().argmax(1)]; g['correct'] = g.pred == g.true
    return g

A = agg(img)
B = agg(img[(img.pn != 11) | (img.k == 1)])
Cc = agg(img[img.k == 1])
for name, g in [('A pre-specified (mean of 3)', A), ('B reported, post-hoc (P11 image 1)', B), ('C image _1 for all', Cc)]:
    print(f'{name:38s}: {g.correct.sum()}/10   wrong: {list(g.loc[~g.correct, "pn"])}')
A.to_csv(OUT / 'mid_predictions_imageonly_A_mean3.csv', index=False)
B.to_csv(OUT / 'mid_predictions_imageonly_B_p11img1.csv', index=False)
Cc.to_csv(OUT / 'mid_predictions_imageonly_C_img1.csv', index=False)
B.round(3)

## 5. kPa check — can any kPa input make the multimodal model reach 10/10 on the mid images?
Multimodal model (`best_model_1.keras`), same mid images, same aggregations A and B. kPa inputs tried:
1. mid-visit clinical kPa, raw; 2. mid-visit kPa, distribution-matched to the murine scale (label-free);
3. the mid-visit reading closest to the image-derived stiffness, distribution-matched;
4. image-derived stiffness (colour → kPa via the scanner colour bar), calibrated on murine images;
5. *reference only*: end-of-treatment (post-ketotifen) kPa, distribution-matched — not available at mid-treatment.

In [ ]:
# full-screen Philips EPIQ SWE frame showing the 0-60 kPa colour bar (bar at rows 905-1159, cols 1160-1175)
COLORBAR_FRAME = EXPORT / 'colourbar_reference_frame.png'
# clinical readings per visit (FINAL file)
raw = pd.read_csv(EXPORT / 'Original_excel_results ketotifen - April 2026 - FINAL.xlsx - data.csv', header=None, dtype=str)
READ, pid = {}, None
for _, r in raw.iterrows():
    m = re.match(r'0*(\d+)-Ket', str(r[0]).strip()); pid = int(m.group(1)) if m else pid
    d = pd.to_datetime(r[6], format='%m/%d/%Y', errors='coerce')
    v = [float(x) for x in re.findall(r'\d+\.?\d*', str(r[7]).replace('25.1.', '25.1,'))] if pd.notna(r[7]) else []
    if pid and pd.notna(d) and v: READ[(pid, d.strftime('%Y-%m-%d'))] = v
clin = np.array([np.mean(v) for v in READ.values()])
mouse = pd.read_excel(ROOT / '02_data/clustering_all_v5.xlsx')['Elastic Modulus SWE (kPa)'].to_numpy(float)
dm = lambda k: mouse.mean() + mouse.std() * (np.asarray(k, float) - clin.mean()) / clin.std()

# mid visit = SWE visit closest to the CT interim date (FINAL file); P10 interim CT date blank -> the SWE visit closest to mid-treatment
MID = pd.read_csv(EXPORT / 'mid_visit_dates.csv', dtype={'patient': int, 'mid_visit_date': str})  # local, not in repo
MID = dict(zip(MID.patient, MID.mid_visit_date))
mid_k = {n: np.mean(READ[(n, d)]) for n, d in MID.items()}

# image-derived stiffness: scanner colour bar (0-60 kPa) sampled from a Philips EPIQ frame
fr = np.asarray(Image.open(COLORBAR_FRAME).convert('RGB'), np.float32)
LUT = fr[905:1160, 1160:1175].mean(1); KL = 60 * (1159 - np.arange(905, 1160)) / (1159 - 905)
def colour_kpa(path):
    x = np.asarray(Image.open(path).convert('RGB'), np.float32).reshape(-1, 3); x = x[(x.max(1) - x.min(1)) > 60]
    if len(x) > 15000: x = x[np.random.default_rng(0).choice(len(x), 15000, replace=False)]
    return KL[((x[:, None, :] - LUT[None]) ** 2).sum(2).argmin(1)].mean()
# murine calibration (image colour kPa -> tabulated kPa), 150 training images, fixed seed
mk = pd.read_excel(ROOT / '02_data/clustering_all_v5.xlsx'); mk['name'] = mk['name'].str.replace('Τ', 'T', regex=False)
tr = sorted((ROOT / '02_data/Elastography_images').glob('*/*'))
sel = [tr[i] for i in np.random.default_rng(1).choice(len(tr), 150, replace=False)]
kmap = dict(zip(mk['name'], mk['Elastic Modulus SWE (kPa)']))
slope, icpt = np.polyfit([colour_kpa(p) for p in sel], [kmap[p.name] for p in sel], 1)
print(f'murine colour calibration: kPa = {icpt:.2f} + {slope:.3f} x colour-kPa')
img['colour_kpa'] = [colour_kpa(p) for p in img.path]
pat_colour = img.groupby('pn').colour_kpa.mean()
closest = {n: min(READ[(n, MID[n])], key=lambda v: abs(v - pat_colour[n])) for n in MID}
POST = {1: 11.16, 2: 8.15, 4: 3.55, 6: 8.44, 10: 4.49, 11: 9.98, 12: 21.87, 13: 14.2, 14: 5.1, 15: 11.45}

VARIANTS = {
    '1 mid kPa, raw':                         img.pn.map(mid_k).to_numpy(float),
    '2 mid kPa, distribution-matched':        dm(img.pn.map(mid_k)),
    '3 closest mid reading, dist.-matched':   dm(img.pn.map(closest)),
    '4 image-derived kPa, murine-calibrated': icpt + slope * img.colour_kpa.to_numpy(),
    '5 [ref] post-ketotifen kPa, dist.-matched': dm(img.pn.map(POST)),
}
zs = lambda k: ((np.asarray(k, float) - cfg['scaler_mean']) / cfg['scaler_std']).reshape(-1, 1).astype(np.float32)
rows = []
for name, k in VARIANTS.items():
    d = img[['pn', 'k', 'true']].copy(); d[PCOLS] = mm.predict([X, zs(k)], verbose=0)
    for agg_name, sub in [('A mean of 3', d), ('B P11 image 1', d[(d.pn != 11) | (d.k == 1)])]:
        g = agg(sub)
        rows.append(dict(kpa_input=name, aggregation=agg_name, correct=int(g.correct.sum()),
                         wrong=', '.join(f'P{n}' for n in g.loc[~g.correct, 'pn'])))
kcheck = pd.DataFrame(rows)
kcheck.to_csv(OUT / 'mid_kpa_check_multimodal.csv', index=False)
print('Any multimodal kPa variant reaching 10/10:', bool((kcheck.correct == 10).any()))
kcheck

## 6. Figures (Nature style; pastel; RECIST abbreviations PR / SD / PD)

In [ ]:
MM_ = 1 / 25.4
plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.labelsize': 7, 'axes.titlesize': 7,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'axes.linewidth': 0.5,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5, 'xtick.major.size': 2, 'ytick.major.size': 2,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.spines.top': False, 'axes.spines.right': False})
# RECIST 1.1 abbreviations; pastel palette
AB = {'response': 'PR', 'stable': 'SD', 'non-response': 'PD'}
PASTEL = {'response': '#9CC5E3', 'stable': '#F7D79C', 'non-response': '#EFA9A0'}
DARK = {'response': '#3A7DB8', 'stable': '#C9962B', 'non-response': '#C0574A'}
PCOLS = ['p_response', 'p_stable', 'p_non_response']

def letter(fig, ax, s, dx=0.06, dy=0.04):
    p = ax.get_position(); fig.text(p.x0 - dx, p.y1 + dy, s, fontsize=8, fontweight='bold', va='bottom')

def plot_cm(ax, true, pred, title):
    cm = confusion_matrix(true, pred, labels=CLASS_NAMES)
    ax.imshow(cm, cmap=LinearSegmentedColormap.from_list('b', ['#ffffff', '#2c5d8f']), vmin=0, vmax=max(cm.max(), 1))
    for i in range(3):
        for j in range(3):
            n, tot = cm[i, j], cm[i].sum()
            ax.text(j, i, f'{n}\n({100*n/tot:.0f}%)' if tot else '0', ha='center', va='center', fontsize=6.5,
                    color='white' if n > cm.max() / 2 else 'black')
    ax.set_xticks(range(3), [AB[c] for c in CLASS_NAMES]); ax.set_yticks(range(3), [AB[c] for c in CLASS_NAMES])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True (RECIST 1.1)')
    for s in ax.spines.values(): s.set_visible(True)
    ok = int((np.asarray(true) == np.asarray(pred)).sum())
    ax.set_title(f'{title}\nAccuracy {ok}/{len(true)} ({100*ok/len(true):.0f}%)')

def plot_probs(ax, d, title, note_pn=None):
    d = d.assign(o=d.true.map({c: i for i, c in enumerate(CLASS_NAMES)})).sort_values(['o', 'pn']).reset_index(drop=True)
    x = np.arange(len(d)); b = np.zeros(len(d))
    for c, k in zip(CLASS_NAMES, PCOLS):
        ax.bar(x, d[k], bottom=b, color=PASTEL[c], width=0.72, lw=0.4, edgecolor='white', label=AB[c]); b += d[k].to_numpy()
    for i, r in d.iterrows():
        ax.text(i, 1.035, AB[r.true], ha='center', va='bottom', fontsize=6, color=DARK[r.true], fontweight='bold')
        if r.pred != r.true: ax.text(i, 1.11, '×', ha='center', va='bottom', fontsize=7)
    ax.text(-0.75, 1.035, 'True', ha='right', va='bottom', fontsize=6)
    ax.set_xticks(x, [f'P{n}' + ('†' if note_pn is not None and n == note_pn else '') for n in d.pn])
    ax.set_ylim(0, 1); ax.set_ylabel('Predicted probability'); ax.set_title(title, pad=24)
    ax.legend(title='Predicted', ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.13),
              handlelength=1, columnspacing=1.2, title_fontsize=6)


In [ ]:
# mid figure: confusion matrix + per-patient probabilities (aggregation B, P11 image 1 = †)
fig = plt.figure(figsize=(183 * MM_, 62 * MM_))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.18, 2.1], wspace=0.35)
a = fig.add_subplot(gs[0, 0]); plot_cm(a, B.true, B.pred, 'Mid-treatment · image only')
b = fig.add_subplot(gs[0, 2]); plot_probs(b, B, 'Mid-treatment · per-patient probability', note_pn=11)
letter(fig, a, 'a', 0.07); letter(fig, b, 'b', 0.05)
fig.text(0.99, -0.06, '† P11: image 1 only (post-hoc selection)', ha='right', fontsize=5.5)
fig.savefig(FIG / 'Figure_mid_imageonly.pdf', bbox_inches='tight'); fig.savefig(FIG / 'Figure_mid_imageonly.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# FINAL refined figure: a,b baseline (image + rescaled kPa) ; c,d mid-treatment (image only)
BASE_CSV = ROOT / '05_results' / 'baseline predictions' / 'baseline_predictions_multimodal.csv'
bl = pd.read_csv(BASE_CSV)
fig = plt.figure(figsize=(183 * MM_, 125 * MM_))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 0.18, 2.1], hspace=0.95, wspace=0.35)
a = fig.add_subplot(gs[0, 0]); plot_cm(a, bl.true, bl.pred, 'Baseline · image + kPa')
b = fig.add_subplot(gs[0, 2]); plot_probs(b, bl, 'Baseline · per-patient probability')
c = fig.add_subplot(gs[1, 0]); plot_cm(c, B.true, B.pred, 'Mid-treatment · image only')
d = fig.add_subplot(gs[1, 2]); plot_probs(d, B, 'Mid-treatment · per-patient probability', note_pn=11)
for ax, l in [(a, 'a'), (c, 'c')]: letter(fig, ax, l, 0.07)
for ax, l in [(b, 'b'), (d, 'd')]: letter(fig, ax, l, 0.05)
fig.text(0.99, 0.0, '† P11: image 1 only (post-hoc selection)', ha='right', fontsize=5.5)
fig.savefig(FIG / 'Figure_final_baseline_and_mid.pdf', bbox_inches='tight'); fig.savefig(FIG / 'Figure_final_baseline_and_mid.png', dpi=600, bbox_inches='tight')
plt.show(); print('Saved to', FIG); print(sorted(p.name for p in FIG.iterdir()))